In [0]:
from arelle import Cntlr, ModelManager
from pyspark.sql.functions import col, lit
from arelle.XbrlConst import parentChild
import zipfile

In [0]:
schema = 'finance_staging'
table_name = 'dim_taxonomy_staging'

In [0]:
dbutils.widgets.text("year", "", "GAAP Version Year")
gaap_year_to_process = dbutils.widgets.get("year")

dbutils.widgets.text("target_catalog", "", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")

In [0]:
gaap_version_processed = f'us-gaap/{gaap_year_to_process}'

In [0]:
output_path = f"/Volumes/operations/finance_staging/taxonomy/us-gaap-{gaap_year_to_process}.zip"

In [0]:
with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall("/Volumes/operations/finance_staging/taxonomy/")

In [0]:
cntlr = Cntlr.Cntlr()
model_manager = ModelManager.initialize(cntlr)

taxonomy_path = f"/Volumes/operations/finance_staging/taxonomy/us-gaap-{gaap_year_to_process}/entire/us-gaap-entryPoint-std-{gaap_year_to_process}.xsd"
model_xbrl = model_manager.load(taxonomy_path)

In [0]:
labels = []

for concept in model_xbrl.qnameConcepts.values():
    label = concept.label()
    
    if label:
        labels.append({
            "concept_qname": str(concept.qname),
            "label_text": label
        })

In [0]:
labels_df = spark.createDataFrame(labels)

In [0]:
presentation = []

rel_set = model_xbrl.relationshipSet(parentChild)

for rel in rel_set.modelRelationships:
    presentation.append({
        "parent": str(rel.fromModelObject.qname),
        "child": str(rel.toModelObject.qname),
        "order": rel.order,
        "linkrole": rel.linkrole
    })

In [0]:
presentation_df = spark.createDataFrame(presentation).createOrReplaceTempView('df')

# balance_sheet_df = presentation_df.filter(
#     presentation_df.linkrole.contains("StatementOfFinancialPosition") ##will need to change this in the future to include more statements
# )

In [0]:
%sql select * from df where child like '%DividendsPayableCurrent%'

In [0]:
presentation_labeled = balance_sheet_df \
      .withColumn("gaap_version", lit(gaap_version_processed))\
    .join(labels_df.withColumnRenamed("concept_qname", "child"),
          on="child",
          how="left") \
    .withColumnRenamed("label_text", "child_label") \
    .join(labels_df.withColumnRenamed("concept_qname", "parent"),
          on="parent",
          how="left") \
    .withColumnRenamed("label_text", "parent_label")\
      

In [0]:
#presentation_labeled.write.mode("overwrite").saveAsTable(f"{target_catalog}.{schema}.{table_name}")